"""
  About this project:
  
  Problem : the project solves the problem to predict if any money transaction is a fraud or legistimate 
  This a standard ML project which will train on the pattern of data including valid and fraudulant transactions patterns 
  used by the robers and hackers  such as most of the fraud transactions are happened at middle or weekends, at midnight hours 
  most of the fraud transactions are related to travel because of high paying amounts .
"""

In [54]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier 
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import pickle 
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score,recall_score, f1_score, roc_auc_score


In [56]:
train_data= pd.read_csv('train_data.csv')
test_data = pd.read_csv('test_data.csv')

In [57]:
train_data['is_fraud'].value_counts()

is_fraud
0    7840
1     160
Name: count, dtype: int64

In [58]:
X_train = train_data.iloc[:,:4]
y_train= train_data['is_fraud']

In [59]:
X_test = train_data.iloc[:,:4]
y_test= train_data['is_fraud']

## Data preprocessing 
- preprocess data to feed model 

In [55]:
scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown='ignore')

### Use pipelining technique :
- benefit : no manual transformation of training and testing data seprately

In [67]:
preprocessor = ColumnTransformer(transformers=[
    ('numerical',scaler,X_train.iloc[:,:3].columns),
    ('categorical',encoder, X_train.iloc[:,3:4].columns)
])

In [ ]:
# manual preprocessing 

# X_train.iloc[:, :3] = scaler.fit_transform(X_train.iloc[:, :3])

# # # encode last columns 
# X_train = pd.get_dummies(X_train, columns=[X_train.columns[-1]], dtype=int)

In [ ]:
# X_test.iloc[:, :3] = scaler.transform(X_test.iloc[:, :3])

# # # encode last columns 
# X_test = pd.get_dummies(X_test, columns=[X_test.columns[-1]], dtype=int)

# Build models

In [70]:
log_reg_model = Pipeline([
    ('Preprocessing',preprocessor),
    ("model",LogisticRegression(penalty = 'l2', solver ='lbfgs', C= 1.0))
     ])

log_reg_model

Pipeline(steps=[('Preprocessing',
                 ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                                  Index(['amount', 'hour', 'day_of_week'], dtype='object')),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['merchant_category'], dtype='object'))])),
                ('model', LogisticRegression())])

In [72]:
rf_model = Pipeline([
    ('Preprocessing',preprocessor),
    ("model",RandomForestClassifier(n_estimators =200,random_state =42))
])
rf_model


Pipeline(steps=[('Preprocessing',
                 ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                                  Index(['amount', 'hour', 'day_of_week'], dtype='object')),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['merchant_category'], dtype='object'))])),
                ('model',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

In [73]:
xgb_model = Pipeline([
    ('Preprocessing',preprocessor),
    ("model",XGBClassifier(n_estimators= 200,learning_rate = 0.1,max_depth =4,use_label_encoder = False,random_state =42))
])

xgb_model

Pipeline(steps=[('Preprocessing',
                 ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                                  Index(['amount', 'hour', 'day_of_week'], dtype='object')),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['merchant_category'], dtype='object'))])),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsampl...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

## Model Training 

In [74]:
log_reg_model.fit(X_train,y_train)
rf_model.fit(X_train,y_train)
xgb_model.fit(X_train,y_train)

D:\miniconda\Lib\site-packages\xgboost\core.py:158: UserWarning: [13:20:57] WARNING: D:\bld\xgboost-split_1738880195977\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Pipeline(steps=[('Preprocessing',
                 ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                                  Index(['amount', 'hour', 'day_of_week'], dtype='object')),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['merchant_category'], dtype='object'))])),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsampl...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

## Model prediction 

prediction on trianing data

In [75]:
log_reg_train_pred =log_reg_model.predict(X_train)
rf_train_pred = rf_model.predict(X_train)
xgb_train_pred = xgb_model.predict(X_train)

prediction on testing data

In [76]:
log_reg_pred =log_reg_model.predict(X_test)
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

## Model Evaluation

In [78]:
result = {}
def evaluate_model(model_name,y_train,y_test,y_train_pred,y_pred):
    print(f'\n{model_name} Evaluation\n')
    train_accuracy =accuracy_score(y_train,y_train_pred)
    test_accuracy =accuracy_score(y_test,y_pred)
    precision =precision_score(y_test,y_pred)
    recall = recall_score(y_test,y_pred)
    f1= f1_score(y_test,y_pred)
   
    # store the all metrices score in dictionary to use in dataframe later
    result[model_name] = {
        'Train_Accuracy' :train_accuracy,
        'Test_Accuracy': test_accuracy,
        'Precision': precision,
        'Recall': recall,
        'f1_score': f1,
    }

    print(f'\nname {model_name}')
    print(f'Train Accuracy: {train_accuracy}')
    print(f'Test Accuracy: {test_accuracy}')
    print(f'Precision score: {precision}')
    print(f'Recall score: {recall}')
    print(f'f1_score : {f1}')
    print('\nConfusion Matrix\n',confusion_matrix(y_test,y_pred))
    print('\n\nClassification Report\n',classification_report(y_test,y_pred))


In [79]:
evaluate_model('Logistic Regression',y_train, y_test, log_reg_train_pred,log_reg_pred)


Logistic Regression Evaluation


name Logistic Regression
Train Accuracy: 0.98275
Test Accuracy: 0.98275
Precision score: 0.775
Recall score: 0.19375
f1_score : 0.31

Confusion Matrix
 [[7831    9]
 [ 129   31]]


Classification Report
               precision    recall  f1-score   support

           0       0.98      1.00      0.99      7840
           1       0.78      0.19      0.31       160

    accuracy                           0.98      8000
   macro avg       0.88      0.60      0.65      8000
weighted avg       0.98      0.98      0.98      8000



In [80]:
evaluate_model('Random Forest',y_train, y_test, rf_train_pred,rf_pred)


Random Forest Evaluation


name Random Forest
Train Accuracy: 1.0
Test Accuracy: 1.0
Precision score: 1.0
Recall score: 1.0
f1_score : 1.0

Confusion Matrix
 [[7840    0]
 [   0  160]]


Classification Report
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      7840
           1       1.00      1.00      1.00       160

    accuracy                           1.00      8000
   macro avg       1.00      1.00      1.00      8000
weighted avg       1.00      1.00      1.00      8000



In [81]:
evaluate_model('XG Boost',y_train, y_test, xgb_train_pred,xgb_pred)


XG Boost Evaluation


name XG Boost
Train Accuracy: 0.988875
Test Accuracy: 0.988875
Precision score: 0.8901098901098901
Recall score: 0.50625
f1_score : 0.6454183266932271

Confusion Matrix
 [[7830   10]
 [  79   81]]


Classification Report
               precision    recall  f1-score   support

           0       0.99      1.00      0.99      7840
           1       0.89      0.51      0.65       160

    accuracy                           0.99      8000
   macro avg       0.94      0.75      0.82      8000
weighted avg       0.99      0.99      0.99      8000



# compare all model stats

In [82]:
result_df = pd.DataFrame(result).T.sort_values(by = 'Test_Accuracy',ascending = False)
result_df

,Train_Accuracy,Test_Accuracy,Precision,Recall,f1_score
Random Forest,1.000000,1.000000,1.00000,1.00000,1.000000
XG Boost,0.988875,0.988875,0.89011,0.50625,0.645418
Logistic Regression,0.982750,0.982750,0.77500,0.19375,0.310000


# Save models using pickle
- entire pipeline along with its transformation preprocessor saved
- benefit : no manual transformation of unseen data when predicting


In [83]:
import pickle 

pickle.dump(log_reg_model, open("Fraud_transaction_logreg_model.pkl",'wb'))
pickle.dump(rf_model, open("Fraud_transaction_randomforest_model.pkl",'wb'))
pickle.dump(xgb_model, open("Fraud_transaction_xgb_model.pkl",'wb'))



# Load a saved model for prediction 

In [84]:
with open('Fraud_transaction_xgb_model.pkl','rb') as file:
    model= pickle.load(file)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [89]:
raw_data = {
    "amount": [1000],
    "hour": [23],
    "day_of_week":[4],
    "merchant_category":['travel']
    
}

In [92]:
unseen_data = pd.DataFrame(raw_data)
unseen_data

,amount,hour,day_of_week,merchant_category
0,1000,23,4,travel


In [98]:
if model.predict(unseen_data)[0] == 1:
    print("Fraud Transaction")
else:
    print('Legitimate Transaction')

Fraud Transaction
